In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import torch
import viser
import matplotlib.pyplot as plt
from scipy.spatial.transform import Rotation
import viser.transforms as vtf

from nerfstudio.data.scene_box import SceneBox
from nerfstudio.cameras.cameras import Cameras
from nerfstudio.utils.spherical_harmonics import SH2RGB
from nerfstudio.cameras.cameras import CameraType

from shadow_splat.model import ShadowSplatModel, ShadowSplatModelConfig
from shadow_splat.util.covariance_utils import compute_cov

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Setup scene and model


In [55]:
config = ShadowSplatModelConfig()
scene_box = SceneBox(aabb=torch.tensor([[-1, -1, -1], [1, 1, 1]]))
model = ShadowSplatModel(config, scene_box, num_train_data=100).to(device)

In [56]:
# Plane of points
plane_xy = torch.stack(
    torch.meshgrid(torch.linspace(-1, 1, 100), torch.linspace(-1, 1, 100)), dim=-1
).reshape(-1, 2)
plane_xyz = torch.cat([plane_xy, torch.zeros_like(plane_xy[..., :1])], dim=-1)

In [57]:
# Create cylinder points
num_surface_points = 100  # Number of points around the cylinder surface
num_layers = 200  # Number of layers in z direction
num_interior_points = 10000  # Number of interior points
radius = 0.25  # Cylinder radius
z_start = 0.0  # Starting z coordinate
z_end = 0.5  # Ending z coordinate

# Create surface points
angles = torch.linspace(0, 2 * torch.pi, num_surface_points)
z_coords = torch.linspace(z_start, z_end, num_layers)
angles_grid, z_grid = torch.meshgrid(angles, z_coords, indexing="ij")
x_coords = radius * torch.cos(angles_grid)
y_coords = radius * torch.sin(angles_grid)
surface_points = torch.stack([x_coords, y_coords, z_grid], dim=-1).reshape(-1, 3)

# Create interior points using rejection sampling
interior_points = []
while len(interior_points) < num_interior_points:
    # Generate random points in a cube
    x = torch.rand(num_interior_points) * 2 * radius - radius
    y = torch.rand(num_interior_points) * 2 * radius - radius
    z = torch.rand(num_interior_points) * (z_end - z_start) + z_start

    # Stack coordinates
    points = torch.stack([x, y, z], dim=-1)

    # Keep only points inside the cylinder (x^2 + y^2 <= radius^2)
    mask = (points[:, 0] ** 2 + points[:, 1] ** 2) <= radius**2
    interior_points.append(points[mask])

interior_points = torch.cat(interior_points, dim=0)[:num_interior_points]

# Combine surface and interior points
cylinder_points = torch.cat([surface_points, interior_points], dim=0)

In [58]:
# Combine
points = torch.cat([plane_xyz, cylinder_points], dim=0)

# Random colors
colors = 255 * torch.ones(points.shape[0], 3)

model.seed_points = (points, colors)

In [59]:
model.populate_modules()
model.training = False
model = model.to(device)

## Camera


In [60]:
# Facing x-axis with z=0.5
R = torch.tensor([[0.0, 0.0, 1.0], [1.0, 0.0, 0.0], [0.0, 1.0, 0.0]])
t = torch.tensor([[1.0], [0.0], [0.5]])

# Combine into camera_to_worlds matrix
camera_to_worlds = torch.cat([R, t], dim=1)
camera_to_worlds = camera_to_worlds.unsqueeze(0)  # Add batch dimension

# Define camera intrinsics
H = 360  # Height
W = 640  # Width
FOV = 60  # Field of view in degrees

# Convert FOV to radians and compute focal length
focal_length = W / (2 * np.tan(np.deg2rad(FOV / 2)))
fx = focal_length
fy = focal_length
cx = W / 2  # Principal point at image center
cy = H / 2

camera = Cameras(
    camera_to_worlds=camera_to_worlds,
    fx=fx,
    fy=fy,
    cx=cx,
    cy=cy,
    width=W,
    height=H,
).to(device)

## Render


In [61]:
outputs = model(camera)

In [ ]:
plt.imshow(outputs["rgb"].detach().cpu())

In [ ]:
viser_server = viser.ViserServer(port=7007)
viser_server.gui.configure_theme(dark_mode=True)

In [64]:
centers = model.means.detach().clone().cpu().numpy()
colors = SH2RGB(model.features_dc.detach().clone().cpu().numpy())
opacities = torch.sigmoid(model.opacities.detach().clone()).cpu().numpy()
scales = torch.exp(model.scales.detach().clone())
covs = compute_cov(model.quats.detach().clone(), scales).cpu().numpy()

In [65]:
splat_handle = viser_server.scene.add_gaussian_splats(
    "splats",
    centers=centers,
    rgbs=colors,
    opacities=opacities,
    covariances=covs,
)

In [ ]:
c2w = camera.camera_to_worlds.cpu().numpy()[0]
R = vtf.SO3.from_matrix(c2w[:3, :3])
R = R @ vtf.SO3.from_x_radians(np.pi)

viser_server.scene.add_camera_frustum(
    name="camera",
    fov=np.deg2rad(FOV),
    aspect=W / H,
    scale=0.1,
    color=(0.2, 0.2, 1.0),
    wxyz=R.wxyz,
    position=c2w[:3, 3],
)

## Light source


In [67]:
R = torch.tensor([[0.0, 0.0, 1.0], [1.0, 0.0, 0.0], [0.0, 1.0, 0.0]])
t = torch.tensor([[2.0], [0.0], [0.5]])

# Combine into camera_to_worlds matrix
camera_to_worlds = torch.cat([R, t], dim=1)
camera_to_worlds = camera_to_worlds.unsqueeze(0)  # Add batch dimension

# Define camera intrinsics
H = 360  # Height
W = 640  # Width
FOV = 60  # Field of view in degrees

# Convert FOV to radians and compute focal length
focal_length = W / (2 * np.tan(np.deg2rad(FOV / 2)))
fx = focal_length
fy = focal_length
cx = W / 2  # Principal point at image center
cy = H / 2

light_source = Cameras(
    camera_to_worlds=camera_to_worlds,
    fx=fx,
    fy=fy,
    cx=cx,
    cy=cy,
    width=W,
    height=H,
    camera_type=CameraType.PERSPECTIVE,
).to(device)

In [ ]:
model.update_light_source(light_source)